# LeetCode 279 — Perfect Squares bằng Python

Link đề: https://leetcode.com/problems/perfect-squares/ · Độ khó: **Medium** · Chủ đề: Breadth-First Search, Math, Dynamic Programming, Knapsack, Unbounded Knapsack

Skeleton LeetCode:

```python
class Solution:
    def numSquares(self, n: int) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập (đệ quy có nhớ trên mọi cách trừ bớt một số chính phương).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **số lượng ít nhất các số chính phương** (`1, 4, 9, 16, ...`, được dùng lặp lại tùy ý) có **tổng bằng `n`**.
2. Nhận ra **thuật toán tham lam "trừ số chính phương lớn nhất" là sai** (ví dụ `12 = 9 + 1 + 1 + 1` cần 4 số nhưng `12 = 4 + 4 + 4` chỉ cần 3).
3. Đi từ **đệ quy thử mọi số chính phương (brute force)** → **quy hoạch động 1 chiều kiểu "balo không giới hạn" `O(n√n)`** (và dạng 2 chiều theo cách tham khảo) → **BFS theo tầng (đường đi ngắn nhất)** → **định lý bốn số chính phương Lagrange + định lý ba số chính phương Legendre (`O(√n)`)** → **kiểm tra "tổng của `k` số chính phương" bằng đệ quy giới hạn độ sâu**.
4. Hiểu **ba cách nhìn cùng một bài toán**: quy hoạch động (chia nhỏ `n`), đồ thị (đỉnh là các số `0..n`, cạnh trừ một số chính phương, tìm đường ngắn nhất), và lý thuyết số (đáp án chỉ có thể là `1, 2, 3` hoặc `4`).
5. Tránh các lỗi kinh điển: tham lam, quên `dp[0] = 0`, BFS không đánh dấu đã thăm (bùng nổ trạng thái), và kiểm tra dạng `4^a(8b+7)` thiếu bước chia `4`.

## 2. Đề bài & ví dụ

Cho số nguyên `n`, hãy trả về **số lượng ít nhất các số chính phương** có tổng bằng `n`.

**Số chính phương** là số nguyên bằng bình phương của một số nguyên (tích của một số nguyên với chính nó). Ví dụ `1, 4, 9, 16` là số chính phương, còn `3` và `11` thì không.

**Ví dụ 1**

- Input: `n = 12`
- Output: `3`
- Giải thích: `12 = 4 + 4 + 4`.

**Ví dụ 2**

- Input: `n = 13`
- Output: `2`
- Giải thích: `13 = 4 + 9`.

**Ràng buộc:** `1 <= n <= 10⁴`.

In [1]:
from typing import List
import random
from math import isqrt
from collections import deque
from functools import cache

examples = [
    (12, 3),
    (13, 2),
]
for n, expected in examples:
    print(f"n={n}, kỳ vọng {expected}")

n=12, kỳ vọng 3
n=13, kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Số chính phương và `math.isqrt`

Các số chính phương đầu tiên: `1, 4, 9, 16, 25, ...` (`k²`). `math.isqrt(n)` trả phần nguyên của `√n` (chính xác, không sai số thực). `n` là số chính phương khi `isqrt(n) ** 2 == n`. Các số chính phương `≤ n` là `1², 2², ..., isqrt(n)²`.

In [2]:
n = 30
print([k * k for k in range(1, isqrt(n) + 1)])        # [1, 4, 9, 16, 25]
print([x for x in range(1, 40) if isqrt(x) ** 2 == x])
print(isqrt(30), isqrt(36), 36 ** 0.5)

[1, 4, 9, 16, 25]
[1, 4, 9, 16, 25, 36]
5 6 6.0


### 3.2. Quy hoạch động (DP) và "bài toán con"

Gọi `f[i]` là số lượng ít nhất số chính phương có tổng `i`. Để tạo tổng `i`, số chính phương **cuối cùng** ta dùng là `j²` (với `j² ≤ i`); phần còn lại là `i − j²` cần `f[i − j²]` số. Vậy:

```
f[i] = min over j (f[i − j²] + 1),   f[0] = 0
```

Các `f[i − j²]` có chỉ số **nhỏ hơn** `i`, nên điền `f` từ `0` lên `n` (tính trước, dùng lại).

In [3]:
n = 12
f = [0] + [float("inf")] * n
for i in range(1, n + 1):
    j = 1
    while j * j <= i:
        f[i] = min(f[i], f[i - j * j] + 1)           # dùng một số chính phương j*j rồi giải i - j*j
        j += 1
print(f)                                              # f[12] = 3

[0, 1, 2, 3, 1, 2, 3, 4, 2, 1, 2, 3, 3]


### 3.3. Đồ thị và BFS (tìm đường ngắn nhất)

Coi mỗi số `0..n` là một **đỉnh**; có **cạnh** từ `x` tới `x − k²` (khi `x − k² ≥ 0`). "Ít số chính phương nhất để có tổng `n`" chính là **số cạnh ít nhất** trên đường đi từ `n` về `0`: bài toán **đường đi ngắn nhất không trọng số**, giải bằng **BFS** (duyệt theo tầng). Dùng `deque` và tập `visited` để không thăm lại một đỉnh.

In [4]:
def bfs_demo(n):
    squares = [k * k for k in range(1, isqrt(n) + 1)]
    queue = deque([n])
    visited = {n}
    level = 0
    while queue:
        level += 1
        for _ in range(len(queue)):                  # duyệt hết một tầng
            x = queue.popleft()
            for s in squares:
                y = x - s
                if y == 0:
                    return level
                if y > 0 and y not in visited:
                    visited.add(y)
                    queue.append(y)
    return -1


print(bfs_demo(12), bfs_demo(13), bfs_demo(1))

3 2 1


### 3.4. Định lý bốn số chính phương của Lagrange

**Mọi số nguyên dương đều là tổng của tối đa bốn số chính phương.** Do đó đáp án của bài **chỉ có thể là `1, 2, 3` hoặc `4`**. Bằng chứng định lý nằm ngoài phạm vi bài này, nhưng ta có thể **kiểm chứng bằng DP** rằng với mọi `n ≤ 10⁴` đáp án `≤ 4`.

In [5]:
n_max = 10000
f = [0] + [10**9] * n_max
for i in range(1, n_max + 1):
    j = 1
    while j * j <= i:
        f[i] = min(f[i], f[i - j * j] + 1)
        j += 1
print("giá trị lớn nhất của đáp án trên 1..10000:", max(f[1:]))
print("phân phối:", {k: f[1:].count(k) for k in (1, 2, 3, 4)})

giá trị lớn nhất của đáp án trên 1..10000: 4
phân phối: {1: 100, 2: 2649, 3: 5586, 4: 1665}


### 3.5. Định lý ba số chính phương của Legendre: khi nào đáp án là 4

Một số nguyên dương `n` **không** thể viết thành tổng của **ba** số chính phương **khi và chỉ khi** `n` có dạng **`4^a · (8b + 7)`** (`a, b ≥ 0`). Với các số dạng này đáp án là `4`. Cách kiểm tra: chia `n` cho `4` chừng nào còn chia hết, rồi xem phần còn lại có `% 8 == 7` không.

In [6]:
def is_four_form(n):
    while n % 4 == 0:
        n //= 4
    return n % 8 == 7


print([x for x in range(1, 130) if is_four_form(x)])      # 7, 15, 23, 28, 31, 39, 47, 55, 60, 63, ...

[7, 15, 23, 28, 31, 39, 47, 55, 60, 63, 71, 79, 87, 92, 95, 103, 111, 112, 119, 124, 127]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ `n = 12`: các số chính phương `≤ 12` là `1, 4, 9`.

Thử **tham lam** (lấy số chính phương lớn nhất trước): `12 − 9 = 3`; `3 − 1 = 2`; `2 − 1 = 1`; `1 − 1 = 0`: dùng `9 + 1 + 1 + 1` — **4 số**. Nhưng `12 = 4 + 4 + 4` chỉ **3 số**. Vậy tham lam sai: lấy số lớn nhất ngay lúc đầu có thể làm phần còn lại khó chia.

Do đó phải **thử các khả năng**: với mỗi số chính phương `s ≤ n` làm số cuối: `1 + (số ít nhất cho n − s)`; chọn nhỏ nhất. Đó là DP.

| `i` | các lựa chọn `f[i − j²] + 1` | `f[i]` |
|---|---|---|
| 1 | `f[0] + 1 = 1` | 1 |
| 2 | `f[1] + 1 = 2` | 2 |
| 3 | `f[2] + 1 = 3` | 3 |
| 4 | `f[3] + 1 = 4`, `f[0] + 1 = 1` | 1 |
| 5 | `f[4] + 1 = 2`, `f[1] + 1 = 2` | 2 |
| 8 | `f[7] + 1`, `f[4] + 1 = 2` | 2 |
| 9 | `f[0] + 1 = 1` | 1 |
| 12 | `f[11] + 1`, `f[8] + 1 = 3`, `f[3] + 1 = 4` | 3 |

In [7]:
def greedy_wrong(n):
    count = 0
    while n:
        n -= isqrt(n) ** 2                       # luôn trừ số chính phương lớn nhất
        count += 1
    return count


for n in (12, 13, 18, 19):
    print(f"n={n}: tham lam = {greedy_wrong(n)}")

n=12: tham lam = 4
n=13: tham lam = 2
n=18: tham lam = 3
n=19: tham lam = 4


## 5. Cách 1 — Brute Force / baseline: đệ quy thử mọi số chính phương

**Ý tưởng.** `solve(m)` = số ít nhất cho tổng `m`: nếu `m == 0` thì `0`; ngược lại `1 + min(solve(m − s))` với mọi số chính phương `s ≤ m`. Viết thẳng từ định nghĩa, **không nhớ** kết quả nên một bài toán con bị tính lại rất nhiều lần. Chỉ dùng được với `n` nhỏ.

In [8]:
def num_squares_recursive(n: int) -> int:
    def solve(m):
        if m == 0:
            return 0
        best = m                                     # tối đa m số 1
        k = 1
        while k * k <= m:
            best = min(best, 1 + solve(m - k * k))   # thử k*k làm một số hạng
            k += 1
        return best

    return solve(n)


for n, expected in examples:
    print(f"n={n} -> {num_squares_recursive(n)} (kỳ vọng {expected})")
print("n=30 ->", num_squares_recursive(30))

n=12 -> 3 (kỳ vọng 3)
n=13 -> 2 (kỳ vọng 2)
n=30 -> 3


**Độ phức tạp (Cách 1).** Không có bộ nhớ: số lời gọi tăng **theo cấp số nhân** theo `n` (mỗi lời gọi sinh `√m` lời gọi con). Với `n = 12` là vài trăm lời gọi, `n = 30` khoảng hàng chục nghìn, `n = 10⁴` hoàn toàn không khả thi. **Space `O(n)`** cho ngăn xếp đệ quy.

**Nút thắt.** Cùng một `solve(m)` được gọi lại rất nhiều lần (từ nhiều đường khác nhau). Chỉ có `n + 1` giá trị `m` khác nhau. Hãy lưu kết quả: đó là DP.

## 6. Ý tưởng thuật toán / bước đột phá

**Bài toán con chồng lặp → DP.** Chỉ có `n + 1` bài toán con `f[0..n]`; mỗi bài mất tối đa `√n` phép thử. Tổng `O(n√n)`; với `n = 10⁴` là `10⁶` phép tính.

**Ba cách nhìn cho cùng một lời giải:**

- **DP (balo không giới hạn):** `f[i] = min_{j² ≤ i} (f[i − j²] + 1)`, `f[0] = 0`. Các "vật" là `1², 2², ...` và dùng lặp lại tùy ý.
- **BFS:** đồ thị đỉnh `0..n`; từ `x` đi tới `x − k²`; số cạnh ít nhất từ `n` về `0` bằng đáp án; BFS tầng-theo-tầng cho đường ngắn nhất.
- **Toán học:** mọi số là tổng của tối đa 4 số chính phương (Lagrange), và ta biết **khi nào** là 1, 2, 3, 4 (Legendre cho trường hợp 4), nên không cần DP: `O(√n)`.

Các cách đều đúng; DP đáng tin cậy nhất, BFS trực quan nhất, toán học nhanh nhất.

## 7. Các cách tiếp theo

### Cách 2 — Quy hoạch động một chiều (balo không giới hạn)

**Giải thích.** `f = [0] + [inf] * n`. Với `i = 1..n` và mỗi `j` có `j² ≤ i`: `f[i] = min(f[i], f[i − j²] + 1)`. Đáp án `f[n]`. (Lời giải tham khảo dùng bảng hai chiều `f[i][j]` = ít số nhất dùng `i` loại số chính phương đầu tiên để ra tổng `j`; ta rút gọn thành một chiều vì vòng lặp ngoài theo `j` hay theo `i` đều cho cùng kết quả khi vật dùng không giới hạn.)

In [9]:
def num_squares_dp(n: int) -> int:
    f = [0] + [float("inf")] * n                      # f[i]: số chính phương ít nhất có tổng i
    for i in range(1, n + 1):
        j = 1
        while j * j <= i:
            f[i] = min(f[i], f[i - j * j] + 1)         # dùng j*j, phần còn lại i - j*j
            j += 1
    return f[n]


for n, expected in examples:
    print(f"n={n} -> {num_squares_dp(n)} (kỳ vọng {expected})")

n=12 -> 3 (kỳ vọng 3)
n=13 -> 2 (kỳ vọng 2)


In [10]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def numSquares(self, n: int) -> int:
        f = [0] + [float("inf")] * n
        for i in range(1, n + 1):
            j = 1
            while j * j <= i:
                f[i] = min(f[i], f[i - j * j] + 1)
                j += 1
        return f[n]

**Độ phức tạp Cách 2:** **Time `O(n√n)`** (với `n = 10⁴`: khoảng `6.7·10⁵` phép thử), **Space `O(n)`**.

### Cách 3 — BFS theo tầng (đường đi ngắn nhất)

**Giải thích.** Hàng đợi bắt đầu từ `n`; mỗi tầng ứng với một số chính phương dùng thêm. Với mỗi `x` trong tầng hiện tại và mỗi `s` chính phương `≤ x`: `y = x − s`; nếu `y == 0` trả số tầng; nếu `y` chưa thăm thì thêm vào tầng sau. Dùng `visited` để mỗi đỉnh vào hàng đợi **một lần**.

In [11]:
def num_squares_bfs(n: int) -> int:
    squares = [k * k for k in range(1, isqrt(n) + 1)]
    queue = deque([n])
    visited = {n}
    level = 0
    while queue:
        level += 1                                      # đang xét các số cần "level" số chính phương
        for _ in range(len(queue)):
            x = queue.popleft()
            for s in squares:
                y = x - s
                if y == 0:
                    return level
                if y < 0:
                    break                                # squares tăng dần: các s sau còn lớn hơn
                if y not in visited:
                    visited.add(y)
                    queue.append(y)
    return -1


for n, expected in examples:
    print(f"n={n} -> {num_squares_bfs(n)} (kỳ vọng {expected})")

n=12 -> 3 (kỳ vọng 3)
n=13 -> 2 (kỳ vọng 2)


**Độ phức tạp Cách 3:** mỗi đỉnh `0..n` vào hàng đợi tối đa một lần, mỗi lần duyệt tối đa `√n` số chính phương: **Time `O(n√n)`** (thường nhanh hơn DP vì dừng sớm khi tới đáp án nhỏ), **Space `O(n)`**.

### Cách 4 — Toán học: Lagrange + Legendre (`O(√n)`)

**Giải thích.** Áp dụng thứ tự:

1. Nếu `n` là số chính phương → **`1`**.
2. Nếu `n` có dạng `4^a (8b + 7)` → **`4`** (định lý Legendre: không viết được thành tổng 3 số chính phương, mà Lagrange cho tối đa 4).
3. Nếu tồn tại `i` sao cho `n − i²` là số chính phương → **`2`** (kiểm tra `i = 1..√n`).
4. Ngược lại → **`3`** (không phải 1, 2, 4 và Lagrange cho ≤ 4, mà `n` không dạng `4^a(8b+7)` nên viết được thành 3 số chính phương).

In [12]:
def num_squares_math(n: int) -> int:
    def is_square(x):
        r = isqrt(x)
        return r * r == x

    if is_square(n):
        return 1                                        # 1 số chính phương
    m = n
    while m % 4 == 0:                                   # bỏ hết thừa số 4
        m //= 4
    if m % 8 == 7:
        return 4                                        # dạng 4^a (8b + 7)
    for i in range(1, isqrt(n) + 1):
        if is_square(n - i * i):
            return 2                                    # n = i^2 + j^2
    return 3


for n, expected in examples:
    print(f"n={n} -> {num_squares_math(n)} (kỳ vọng {expected})")

n=12 -> 3 (kỳ vọng 3)
n=13 -> 2 (kỳ vọng 2)


**Độ phức tạp Cách 4:** **Time `O(√n)`** (chia 4 tối đa `O(log n)` lần, và vòng lặp `i` tối đa `√n`), **Space `O(1)`**. Nhanh nhất nhưng dựa vào hai định lý (không tự chứng minh trong bài).

### Cách 5 — Kiểm tra "là tổng của đúng `k` số chính phương" (tăng dần `k`)

**Giải thích.** Biết đáp án `≤ 4`, thử `k = 1, 2, 3, 4`: `can(m, k)` kiểm tra `m` có là tổng của **đúng `k`** số chính phương (cho phép số `0`? ta cho phép bằng cách: tổng của tối đa `k` số khác không) bằng đệ quy giới hạn độ sâu `k`: `can(m, 1) = m là số chính phương`; `can(m, k) = ∃ s: can(m − s, k − 1)`. Đáp án là `k` nhỏ nhất thỏa. Có thể dùng `set` các số chính phương để kiểm tra nhanh.

In [13]:
def num_squares_iterative_deepening(n: int) -> int:
    squares = [k * k for k in range(1, isqrt(n) + 1)]
    square_set = set(squares)

    def can(m, k):
        """m có phải tổng của đúng k số chính phương (mỗi số >= 1) không?"""
        if k == 1:
            return m in square_set
        for s in squares:
            if s >= m:
                break
            if can(m - s, k - 1):
                return True
        return False

    for k in range(1, 5):                               # theo Lagrange đáp án <= 4
        if can(n, k):
            return k
    return 4


for n, expected in examples:
    print(f"n={n} -> {num_squares_iterative_deepening(n)} (kỳ vọng {expected})")

n=12 -> 3 (kỳ vọng 3)
n=13 -> 2 (kỳ vọng 2)


**Độ phức tạp Cách 5:** tối đa `√n` lựa chọn mỗi cấp, độ sâu `≤ 4`: tệ nhất `O(n^{2})` (với `k = 3`, đã bị cắt nhờ `s ≥ m`); thực tế rất nhanh với `n ≤ 10⁴` (vài chục nghìn phép thử). **Space `O(√n)`** cho danh sách số chính phương và `set`.

> Cách 2 là cách chắc chắn (DP). Cách 3 là cách đồ thị trực quan. Cách 4 nhanh nhất nhờ lý thuyết số. Cách 1 là chuẩn đối chiếu cho `n` nhỏ.

## 8. TRACE từng bước

Ta theo dõi Cách 2 (DP) cho `n = 13` và Cách 3 (BFS) cho `n = 12`.

In [14]:
def trace_dp(n):
    f = [0] + [float("inf")] * n
    print(f"n={n}: f[0]=0")
    for i in range(1, n + 1):
        options = []
        j = 1
        while j * j <= i:
            options.append(f"f[{i - j * j}]+1={f[i - j * j] + 1}")
            f[i] = min(f[i], f[i - j * j] + 1)
            j += 1
        print(f"  f[{i:2}] = min({', '.join(options)}) = {f[i]}")
    return f[n]


assert trace_dp(13) == 2

n=13: f[0]=0
  f[ 1] = min(f[0]+1=1) = 1
  f[ 2] = min(f[1]+1=2) = 2
  f[ 3] = min(f[2]+1=3) = 3
  f[ 4] = min(f[3]+1=4, f[0]+1=1) = 1
  f[ 5] = min(f[4]+1=2, f[1]+1=2) = 2
  f[ 6] = min(f[5]+1=3, f[2]+1=3) = 3
  f[ 7] = min(f[6]+1=4, f[3]+1=4) = 4
  f[ 8] = min(f[7]+1=5, f[4]+1=2) = 2
  f[ 9] = min(f[8]+1=3, f[5]+1=3, f[0]+1=1) = 1
  f[10] = min(f[9]+1=2, f[6]+1=4, f[1]+1=2) = 2
  f[11] = min(f[10]+1=3, f[7]+1=5, f[2]+1=3) = 3
  f[12] = min(f[11]+1=4, f[8]+1=3, f[3]+1=4) = 3
  f[13] = min(f[12]+1=4, f[9]+1=2, f[4]+1=2) = 2


In [15]:
def trace_bfs(n):
    squares = [k * k for k in range(1, isqrt(n) + 1)]
    queue = deque([n])
    visited = {n}
    level = 0
    print(f"n={n}, các số chính phương = {squares}")
    while queue:
        level += 1
        current = list(queue)
        nxt = []
        for _ in range(len(queue)):
            x = queue.popleft()
            for s in squares:
                y = x - s
                if y == 0:
                    print(f"  tầng {level}: từ {current} trừ {s} từ {x} -> 0: TÌM THẤY, đáp án {level}")
                    return level
                if y > 0 and y not in visited:
                    visited.add(y)
                    queue.append(y)
                    nxt.append(y)
        print(f"  tầng {level}: các số xét {current} -> tầng sau {nxt}")
    return -1


assert trace_bfs(12) == 3

n=12, các số chính phương = [1, 4, 9]
  tầng 1: các số xét [12] -> tầng sau [11, 8, 3]
  tầng 2: các số xét [11, 8, 3] -> tầng sau [10, 7, 2, 4]
  tầng 3: từ [10, 7, 2, 4] trừ 4 từ 4 -> 0: TÌM THẤY, đáp án 3


Bảng tóm tắt (BFS, `n = 12`, số chính phương `1, 4, 9`):

| tầng | các số đang xét | các số sinh ra (chưa thăm) | tìm thấy `0`? |
|---|---|---|---|
| 1 | `[12]` | `11, 8, 3` | không |
| 2 | `[11, 8, 3]` | `10, 7, 2, 4` (từ 11: `10, 7, 2`; từ 8: `7` (đã thăm), `4`; từ 3: `2` (đã thăm)) | không |
| 3 | `[10, 7, 2, 4]` | từ `4`: `4 − 4 = 0` | **có** → đáp án `3` |

Đường đi ngắn nhất `12 → 8 → 4 → 0` tương ứng `12 = 4 + 4 + 4`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (DP một chiều), đơn giản và chắc chắn. Đúng class `Solution`, đúng method `numSquares(self, n: int) -> int`.

In [16]:
class Solution:
    def numSquares(self, n: int) -> int:
        f = [0] + [float("inf")] * n                # f[i]: số chính phương ít nhất có tổng i; f[0] = 0
        for i in range(1, n + 1):
            j = 1
            while j * j <= i:                       # mọi số chính phương j*j <= i
                f[i] = min(f[i], f[i - j * j] + 1)
                j += 1
        return f[n]


for n, expected in examples:
    assert Solution().numSquares(n) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Cấu trúc con tối ưu:** gọi `s` là một số chính phương trong cách chia tối ưu của `i` (tùy ý, ví dụ số cuối cùng). Phần còn lại là một cách chia `i − s` thành các số chính phương; nếu có cách chia `i − s` bằng ít số hơn thì thay vào được cách chia `i` tốt hơn, mâu thuẫn. Vậy `f[i] = 1 + f[i − s]` với `s` đúng và tối ưu hóa trên mọi `s ≤ i` cho `f[i] = min_s (1 + f[i − s])`.
- **Thứ tự điền:** `f[i − s]` có `i − s < i` nên đã được tính. `f[0] = 0` (tổng `0` cần `0` số).
- **Luôn có lời giải:** `i` là tổng của `i` số `1`, nên `f[i] ≤ i`; DP luôn cho giá trị hữu hạn.
- **BFS:** đồ thị có cạnh `x → x − k²` không trọng số; BFS tìm đường ngắn nhất từ `n` đến `0` (đỉnh đầu tiên chạm `0` ở tầng nhỏ nhất). `visited` đúng vì tới một đỉnh lần đầu ở tầng nhỏ nhất.
- **Toán học:** Lagrange: mọi số là tổng của ≤ 4 số chính phương, nên đáp án ∈ {1, 2, 3, 4}. `1` ⇔ số chính phương. `2` ⇔ `n = i² + j²` với `i, j ≥ 1` (khi không phải số chính phương). Legendre: `n` không phải tổng 3 số chính phương ⇔ `n = 4^a(8b + 7)`; với các số đó (khác chính phương và khác tổng hai số chính phương, vì chúng không viết được bằng ≤ 3 số) đáp án là `4`. Còn lại, không phải 1, không phải 2 và không phải dạng cấm: đáp án `3`.
- **Cách 5:** vét cạn theo độ sâu `k ≤ 4`; đúng nhờ Lagrange (chắc chắn tìm thấy với `k ≤ 4`).

## 11. Phân tích Time / Space Complexity

Gọi `n` là số cần phân tích.

- **Cách 1 (đệ quy thuần):** Time tăng theo cấp số nhân (không dùng được cho `n` lớn), Space `O(n)`.
- **Cách 2 (DP một chiều):** Time `O(n√n)`, Space `O(n)`.
- **Cách 3 (BFS):** Time `O(n√n)` (tệ nhất), Space `O(n)`.
- **Cách 4 (Lagrange + Legendre):** Time `O(√n)`, Space `O(1)`.
- **Cách 5 (độ sâu `k ≤ 4`):** Time tối đa `O(n²)` nhưng thực tế nhỏ (cắt tỉa), Space `O(√n)`.
- Với `n ≤ 10⁴`: DP khoảng `6.7·10⁵` phép thử; mọi cách (trừ Cách 1) chạy tức thì.

## 12. Bộ test

Ta gom các cách (trừ Cách 1 cho `n` nhỏ) vào `impls`. `oracle(n)` là **đệ quy có nhớ** thử mọi số chính phương làm số hạng đầu tiên (`functools.cache`), viết độc lập với DP bảng. Với `n` lớn ta so sánh các cách nhanh với nhau và với bảng DP, đồng thời kiểm tra các định lý.

In [17]:
sol = Solution()
impls = {
    "dp": num_squares_dp,
    "bfs": num_squares_bfs,
    "math": num_squares_math,
    "deepening": num_squares_iterative_deepening,
    "submit": sol.numSquares,
}


@cache
def oracle(n):
    if n == 0:
        return 0
    return 1 + min(oracle(n - k * k) for k in range(1, isqrt(n) + 1))


for m in range(1, 10001):                 # điền bộ nhớ theo thứ tự tăng để đệ quy không bị quá sâu (mỗi lời gọi chỉ còn độ sâu 1)
    oracle(m)

for n, expected in examples:
    assert oracle(n) == expected
    assert num_squares_recursive(n) == expected
    for name, fn in impls.items():
        assert fn(n) == expected, (name, n, fn(n), expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [18]:
def check_all(n, expected, label):
    assert oracle(n) == expected, (label, "oracle", oracle(n), expected)
    for name, fn in impls.items():
        assert fn(n) == expected, (label, name, fn(n), expected)
    print("PASS", label, "->", expected)


check_all(1, 1, "n = 1 nhỏ nhất")
check_all(2, 2, "n = 2 = 1 + 1")
check_all(3, 3, "n = 3 = 1 + 1 + 1")
check_all(4, 1, "n = 4 là số chính phương")
check_all(7, 4, "n = 7 = 4 + 1 + 1 + 1 (dạng 8b + 7)")
check_all(12, 3, "ví dụ 1: tham lam sẽ cho 4")
check_all(13, 2, "ví dụ 2")
check_all(28, 4, "n = 28 = 4 * 7: dạng 4^a (8b + 7)")
check_all(10000, 1, "n = 10^4 = 100^2")
check_all(9999, 4, "n = 9999 = 8*1249 + 7 (dạng 8b + 7)")
check_all(9998, 3, "n = 9998")
check_all(8192, 2, "n = 8192 = 2^13 = 64^2 + 64^2")
check_all(9990, 3, "n = 9990")
print("phân bố n <= 10^4:", {k: sum(1 for m in range(1, 10001) if oracle(m) == k) for k in (1, 2, 3, 4)})

PASS n = 1 nhỏ nhất -> 1
PASS n = 2 = 1 + 1 -> 2
PASS n = 3 = 1 + 1 + 1 -> 3
PASS n = 4 là số chính phương -> 1
PASS n = 7 = 4 + 1 + 1 + 1 (dạng 8b + 7) -> 4
PASS ví dụ 1: tham lam sẽ cho 4 -> 3
PASS ví dụ 2 -> 2
PASS n = 28 = 4 * 7: dạng 4^a (8b + 7) -> 4


PASS n = 10^4 = 100^2 -> 1


PASS n = 9999 = 8*1249 + 7 (dạng 8b + 7) -> 4


PASS n = 9998 -> 3
PASS n = 8192 = 2^13 = 64^2 + 64^2 -> 2


PASS n = 9990 -> 3
phân bố n <= 10^4: {1: 100, 2: 2649, 3: 5586, 4: 1665}


### 12.2. Tính chất và phản ví dụ

In [19]:
random.seed(279)

# Tính chất 1: đáp án luôn thuộc {1, 2, 3, 4} (Lagrange); = 1 <=> số chính phương
table = [0] + [oracle(m) for m in range(1, 10001)]
assert set(table[1:]) <= {1, 2, 3, 4}
assert all((table[m] == 1) == (isqrt(m) ** 2 == m) for m in range(1, 10001))
print("PASS tính chất: đáp án thuộc {1, 2, 3, 4}; bằng 1 chính xác khi n là số chính phương")

# Tính chất 2 (Legendre): đáp án = 4 <=> n = 4^a (8b + 7)
assert all((table[m] == 4) == is_four_form(m) for m in range(1, 10001))
print("PASS tính chất: đáp án = 4 chính xác khi n có dạng 4^a (8b + 7)")

# Tính chất 3: f(m + k^2) <= f(m) + 1 và f(4m) <= f(m) (nhân đôi cạnh hình vuông); f(a + b) <= f(a) + f(b)
for _ in range(500):
    m = random.randint(1, 2000)
    k = random.randint(1, 50)
    assert table[m + k * k] <= table[m] + 1
    assert table[4 * m] <= table[m]
    a, b = random.randint(1, 5000), random.randint(1, 5000)
    assert table[a + b] <= table[a] + table[b]
print("PASS tính chất: f(m + k^2) <= f(m) + 1, f(4m) <= f(m), f(a + b) <= f(a) + f(b)")

# Phản ví dụ: tham lam
bad = [n for n in range(1, 60) if greedy_wrong(n) != table[n]]
print("các n < 60 mà tham lam sai:", bad)
assert 12 in bad and greedy_wrong(12) == 4 and table[12] == 3

PASS tính chất: đáp án thuộc {1, 2, 3, 4}; bằng 1 chính xác khi n là số chính phương
PASS tính chất: đáp án = 4 chính xác khi n có dạng 4^a (8b + 7)
PASS tính chất: f(m + k^2) <= f(m) + 1, f(4m) <= f(m), f(a + b) <= f(a) + f(b)
các n < 60 mà tham lam sai: [12, 18, 19, 22, 23, 32, 41, 42, 43, 48, 52, 56]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [20]:
random.seed(2026)
for n in range(1, 1501):                                   # mọi n từ 1 đến 1500
    expected = oracle(n)
    for name, fn in impls.items():
        assert fn(n) == expected, (name, n, fn(n), expected)
for n in range(1, 31):                                      # brute force đệ quy thuần cho n nhỏ
    assert num_squares_recursive(n) == oracle(n)
for trial in range(300):                                    # n ngẫu nhiên tới 10^4
    n = random.randint(1, 10000)
    expected = oracle(n)
    for name, fn in impls.items():
        assert fn(n) == expected, (name, n)
print("PASS mọi n = 1..1500 và 300 giá trị ngẫu nhiên tới 10^4: mọi cách khớp oracle độc lập; đệ quy thuần khớp n <= 30")

PASS mọi n = 1..1500 và 300 giá trị ngẫu nhiên tới 10^4: mọi cách khớp oracle độc lập; đệ quy thuần khớp n <= 30


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Thuật toán tham lam (luôn trừ số chính phương lớn nhất)

In [21]:
for n in (12, 18, 19, 22):
    print(f"n={n}: tham lam = {greedy_wrong(n)} | đúng = {Solution().numSquares(n)}")

n=12: tham lam = 4 | đúng = 3
n=18: tham lam = 3 | đúng = 2
n=19: tham lam = 4 | đúng = 3
n=22: tham lam = 4 | đúng = 3


**Vì sao sai:** lấy số chính phương lớn nhất trước có thể để lại phần dư khó chia. `12 = 9 + 1 + 1 + 1` (4 số) nhưng tối ưu `4 + 4 + 4` (3 số). DP thử **mọi** số chính phương làm số hạng.

### Lỗi 2: Quên điều kiện cơ sở `f[0] = 0` (khởi tạo mọi giá trị bằng vô cùng)

In [22]:
def sai_khong_co_so(n):
    f = [float("inf")] * (n + 1)                      # sai: thiếu f[0] = 0
    for i in range(1, n + 1):
        j = 1
        while j * j <= i:
            f[i] = min(f[i], f[i - j * j] + 1)
            j += 1
    return f[n]


print("n=12 -> sai:", sai_khong_co_so(12), "| đúng:", Solution().numSquares(12))

n=12 -> sai: inf | đúng: 3


**Vì sao sai:** `f[0]` là điểm xuất phát (không cần số nào để có tổng `0`). Nếu để `inf` thì mọi `f[i]` đều `inf` (`inf + 1 = inf`). Khởi tạo `f[0] = 0`.

### Lỗi 3: BFS không đánh dấu đã thăm

In [23]:
def bfs_khong_visited(n, max_states=200000):
    squares = [k * k for k in range(1, isqrt(n) + 1)]
    queue = deque([n])
    level, processed = 0, 0
    while queue:
        level += 1
        for _ in range(len(queue)):
            x = queue.popleft()
            processed += 1
            if processed > max_states:
                return None, processed
            for s in squares:
                y = x - s
                if y == 0:
                    return level, processed
                if y > 0:
                    queue.append(y)                    # sai: không kiểm tra đã thăm
    return -1, processed


for n in (12, 30, 60):
    print(f"n={n}: BFS không visited ->", bfs_khong_visited(n), "| BFS đúng:", num_squares_bfs(n))

n=12: BFS không visited -> (3, 9) | BFS đúng: 3
n=30: BFS không visited -> (3, 8) | BFS đúng: 3
n=60: BFS không visited -> (4, 50) | BFS đúng: 4


**Vì sao sai (chậm):** không `visited` thì cùng một số `y` được thêm vào hàng đợi **rất nhiều lần** (từ nhiều đường), số trạng thái bùng nổ theo tầng. Đáp án vẫn có thể đúng với `n` nhỏ nhưng với `n` lớn chạy cực chậm (ở đây dừng nhờ giới hạn trạng thái).

### Lỗi 4: Kiểm tra dạng `4^a (8b + 7)` mà quên chia hết cho `4`

In [24]:
def sai_dang_8b7(n):
    def is_square(x):
        r = isqrt(x)
        return r * r == x

    if is_square(n):
        return 1
    if n % 8 == 7:                                       # sai: thiếu bước chia hết cho 4
        return 4
    for i in range(1, isqrt(n) + 1):
        if is_square(n - i * i):
            return 2
    return 3


bad = [n for n in range(1, 200) if sai_dang_8b7(n) != Solution().numSquares(n)]
print("các n < 200 mà cách sai cho kết quả sai:", bad[:10])
print("n=28 -> sai:", sai_dang_8b7(28), "| đúng:", Solution().numSquares(28))

các n < 200 mà cách sai cho kết quả sai: [28, 60, 92, 112, 124, 156, 188]
n=28 -> sai: 3 | đúng: 4


**Vì sao sai:** định lý Legendre áp dụng cho `n = 4^a (8b + 7)`, nên phải chia `n` cho `4` chừng nào còn chia hết **trước** khi xét `% 8 == 7`. `28 = 4 · 7`: `28 % 8 = 4` nên bị coi là "không phải dạng cấm" và cho `3`, nhưng đáp án đúng là `4`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Số lượng ít nhất các số (từ một tập) có tổng bằng `n`, dùng lặp lại": **DP kiểu đổi tiền (coin change) / balo không giới hạn**: `f[i] = min(f[i − c] + 1)`.
- Cũng là **BFS đường ngắn nhất** trên đồ thị `x → x − c`; BFS hữu ích khi đáp án nhỏ.
- Tham lam (lấy số lớn nhất) **không** đúng với tập tùy ý (chỉ đúng với hệ tiền đặc biệt).
- Với số chính phương có định lý: tổng của **≤ 4** số chính phương (Lagrange); dạng `4^a(8b + 7)` cần đúng 4 (Legendre).
- Khởi tạo DP: `f[0] = 0` và các giá trị khác là `inf`.
- Test: đối chiếu với đệ quy có nhớ cho `n` nhỏ và nhiều cách với nhau cho `n` lớn; kiểm tra các định lý trên toàn bộ `n ≤ 10⁴`.

## 15. Biến thể của bài

1. **Coin Change (LeetCode 322):** tập mệnh giá bất kỳ.
2. **Perfect Squares với số lập phương** hoặc lũy thừa `p` bất kỳ.
3. **Số hạng phân biệt** (mỗi số chính phương dùng tối đa một lần).
4. **Liệt kê một cách chia tối ưu** (không chỉ số lượng).
5. **Đếm số cách** chia `n` thành số chính phương (không chỉ tối thiểu).

In [25]:
def min_count_power(n, p):
    """Biến thể 2: ít nhất bao nhiêu số dạng k^p (p >= 2) có tổng n?"""
    powers = []
    k = 1
    while k ** p <= n:
        powers.append(k ** p)
        k += 1
    f = [0] + [float("inf")] * n
    for i in range(1, n + 1):
        for s in powers:
            if s > i:
                break
            f[i] = min(f[i], f[i - s] + 1)
    return f[n]


def min_squares_with_witness(n):
    """Biến thể 4: trả về (số lượng, danh sách số chính phương)."""
    f = [0] + [float("inf")] * n
    choice = [0] * (n + 1)
    for i in range(1, n + 1):
        j = 1
        while j * j <= i:
            if f[i - j * j] + 1 < f[i]:
                f[i] = f[i - j * j] + 1
                choice[i] = j * j
            j += 1
    parts, m = [], n
    while m:
        parts.append(choice[m])
        m -= choice[m]
    return f[n], parts


def count_ways(n):
    """Biến thể 5: số cách chia n thành số chính phương (không phân biệt thứ tự)."""
    ways = [1] + [0] * n
    for k in range(1, isqrt(n) + 1):
        s = k * k
        for i in range(s, n + 1):
            ways[i] += ways[i - s]
    return ways[n]


def distinct_squares_min(n):
    """Biến thể 3: mỗi số chính phương dùng tối đa một lần (balo 0/1); inf nếu không thể."""
    f = [0] + [float("inf")] * n
    for k in range(1, isqrt(n) + 1):
        s = k * k
        for i in range(n, s - 1, -1):              # duyệt ngược để mỗi số chỉ dùng một lần
            f[i] = min(f[i], f[i - s] + 1)
    return f[n]


assert min_count_power(12, 2) == 3 and min_count_power(27, 3) == 1 and min_count_power(16, 3) == 2    # 16 = 8 + 8
assert min_squares_with_witness(12) == (3, [4, 4, 4]) or min_squares_with_witness(12)[0] == 3
cnt, parts = min_squares_with_witness(13)
assert cnt == 2 and sum(parts) == 13 and all(isqrt(p) ** 2 == p for p in parts)
assert count_ways(4) == 2 and count_ways(12) == 5      # 12: 9+1+1+1, 4+4+4, 4+4+1*4, 4+1*8, 1*12
assert distinct_squares_min(14) == 3 and distinct_squares_min(2) == float("inf")      # 14 = 1 + 4 + 9
random.seed(15)
for _ in range(100):
    n = random.randint(1, 300)
    c, p_ = min_squares_with_witness(n)
    assert c == oracle(n) and sum(p_) == n and len(p_) == c
print("PASS biến thể: lũy thừa p, chứng cứ, đếm cách, số hạng phân biệt")

PASS biến thể: lũy thừa p, chứng cứ, đếm cách, số hạng phân biệt


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy thuần | thử mọi số chính phương, không nhớ | cấp số nhân | `O(n)` | oracle, `n` nhỏ |
| 2. DP một chiều | `f[i] = min(f[i − j²] + 1)` | `O(n√n)` | `O(n)` | **cách nộp bài**, an toàn |
| 3. BFS theo tầng | đường ngắn nhất `n → 0` | `O(n√n)` | `O(n)` | trực quan, dừng sớm khi đáp án nhỏ |
| 4. Lagrange + Legendre | đáp án ∈ {1,2,3,4}, kiểm tra theo thứ tự | **`O(√n)`** | `O(1)` | nhanh nhất; cần biết định lý |
| 5. Độ sâu `k ≤ 4` | `can(m, k)` đệ quy, `set` số chính phương | nhỏ trên thực tế | `O(√n)` | minh họa vét cạn có cắt tỉa |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `isqrt(50)`, `[k * k for k in range(1, 5)]`, `Solution().numSquares(9)`, `Solution().numSquares(7)`, `Solution().numSquares(8)`.

**Bài 2 (tính tay).** Lập bảng DP `f[0..10]` bằng tay (dùng số chính phương `1, 4, 9`). Với `n = 10` chỉ ra cách chia. Với `n = 15`: tại sao đáp án là `4`?

**Bài 3 (điền TODO).** Hoàn thành `squares_todo(n)` theo Cách 2 (DP).

In [26]:
def squares_todo(n):
    # TODO 1: f = [0] + [inf] * n
    # TODO 2: với i = 1..n và mỗi j có j*j <= i: f[i] = min(f[i], f[i - j*j] + 1)
    # TODO 3: trả f[n]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `squares_bfs2(n)` theo Cách 3 (BFS theo tầng với `visited`). Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `coin_change(coins, amount)` (LeetCode 322, trả `-1` nếu không thể) bằng DP một chiều. (b) Viết `is_sum_of_two_squares(n)` trả `True` nếu `n = a² + b²` với `a, b ≥ 0`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `isqrt` là căn nguyên dưới; `9` là số chính phương; `7 = 4 + 1 + 1 + 1`. Bài 2: mỗi `f[i]` lấy nhỏ nhất của `f[i − 1] + 1`, `f[i − 4] + 1`, `f[i − 9] + 1`.
- **Mức 2 (vừa):** Bài 1: `7`, `[1, 4, 9, 16]`, `1`, `4`, `2`. Bài 2: `f = [0,1,2,3,1,2,3,4,2,1,2]`: `n = 10 = 9 + 1`. `n = 15 = 4 + 9 + 1 + 1`; `15 = 8·1 + 7` có dạng `8b + 7` nên cần 4. Bài 3: ba TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `queue = deque([n])`, `visited = {n}`, đếm tầng; với mỗi `x` trừ từng số chính phương `s`; `y == 0` thì trả tầng. Bài 5(a): `f[0] = 0`, `f[a] = min(f[a − c] + 1)` cho `c ≤ a`; trả `-1` nếu `inf`. Bài 5(b): thử `a` từ `0` đến `isqrt(n)` và kiểm tra `n − a²` là số chính phương.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert isqrt(50) == 7 and [k * k for k in range(1, 5)] == [1, 4, 9, 16]
assert Solution().numSquares(9) == 1 and Solution().numSquares(7) == 4 and Solution().numSquares(8) == 2

# Bài 2
assert [oracle(i) for i in range(0, 11)] == [0, 1, 2, 3, 1, 2, 3, 4, 2, 1, 2]
assert oracle(15) == 4 and is_four_form(15)

# Bài 3
def squares_todo_ans(n):
    f = [0] + [float("inf")] * n
    for i in range(1, n + 1):
        j = 1
        while j * j <= i:
            f[i] = min(f[i], f[i - j * j] + 1)
            j += 1
    return f[n]

for n in range(1, 300):
    assert squares_todo_ans(n) == oracle(n)

# Bài 4
def squares_bfs2(n):
    squares = [k * k for k in range(1, isqrt(n) + 1)]
    queue, visited, level = deque([n]), {n}, 0
    while queue:
        level += 1
        for _ in range(len(queue)):
            x = queue.popleft()
            for s in squares:
                y = x - s
                if y == 0:
                    return level
                if y > 0 and y not in visited:
                    visited.add(y)
                    queue.append(y)

for n, expected in examples:
    assert squares_bfs2(n) == expected
assert all(squares_bfs2(n) == oracle(n) for n in range(1, 300))

# Bài 5(a), 5(b)
def coin_change(coins, amount):
    f = [0] + [float("inf")] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                f[a] = min(f[a], f[a - c] + 1)
    return -1 if f[amount] == float("inf") else f[amount]

def is_sum_of_two_squares(n):
    return any(isqrt(n - a * a) ** 2 == n - a * a for a in range(0, isqrt(n) + 1))

assert coin_change([1, 2, 5], 11) == 3 and coin_change([2], 3) == -1 and coin_change([1], 0) == 0
assert coin_change([k * k for k in range(1, 101)], 12) == oracle(12)
assert is_sum_of_two_squares(13) and is_sum_of_two_squares(25) and not is_sum_of_two_squares(3) and is_sum_of_two_squares(0)
assert all(is_sum_of_two_squares(n) == (oracle(n) <= 2) for n in range(1, 500))
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (DP kiểu đổi tiền với các số chính phương)

```python
class Solution:
    def numSquares(self, n: int) -> int:
        f = [0] + [float("inf")] * n
        for i in range(1, n + 1):
            j = 1
            while j * j <= i:
                f[i] = min(f[i], f[i - j * j] + 1)
                j += 1
        return f[n]
```

**Checklist**

- [ ] Không dùng tham lam (trừ số chính phương lớn nhất)?
- [ ] `f[0] = 0`, các giá trị khác khởi tạo `inf` (hoặc `i`)?
- [ ] Vòng lặp `j` với điều kiện `j * j <= i` (bao gồm cả `j * j == i`)?
- [ ] Đáp án `f[n]`; luôn `≤ 4` (có thể dùng để kiểm tra nhanh)?
- [ ] Nếu dùng BFS: có `visited` để mỗi đỉnh vào hàng đợi một lần?
- [ ] Nếu dùng toán học: chia hết cho `4` **trước** khi kiểm tra `% 8 == 7`?
- [ ] Đã thử: `n = 1`, số chính phương, `n = 7`, `n = 12`, `n = 28`, `n = 10⁴`?
- [ ] Đã đối chiếu với đệ quy có nhớ trên mọi `n` nhỏ và nhiều cách với nhau trên `n` lớn?
- [ ] Skeleton đúng `class Solution` / `def numSquares(self, n: int) -> int`?